# Appendix 1: Error Analysis of the Fine-tuned DINOv3 Classifier

## Why error analysis
Aggregate metrics (Acc 97.3%, F1 97.3%, AUC 99.5%) hide *where* the model fails. This notebook moves beyond them to dissect the 175 False Negatives and 27 False Positives on `test_balanced.csv`, identify the failure sub-populations (method, domain, class, image characteristics), and produce a qualitative gallery for manual inspection.

**Single variable changed**: none — this is an analysis-only pass over the already-trained checkpoint.
**Held constant**: the fine-tuned `dinov3_finetuned_balanced_best.pt` checkpoint and `test_balanced.csv` (leakage-free identity-disjoint test set).

## Roadmap Table

| Step | Description | What it does | Import path |
|---|---|---|---|
| 1 | Setup: paths, device, seed, artifact dirs | Define checkpoint/test paths; make output dirs | `src.utils.seeding`, `pathlib` |
| 2 | Load model + checkpoint, run predictions once | Rebuild DINOv3 classifier, load weights, predict on test; persist NPZ/CSV | `src.models.dinov3_vit`, `src.training.train` |
| 3 | Confusion matrix deep dive | Plot full CM; isolate FN/FP; sanity-check vs report | `matplotlib`, `sklearn` |
| 4 | Error categorization by metadata | Group errors by method/domain/label from `test_data_v3/manifest.csv` | `pandas` |
| 5 | Statistical breakdown | Per-class and per-method precision/recall/F1 tables | `sklearn.metrics` |
| 6 | Qualitative analysis | Sample + grid-view misclassified images for manual inspection | `matplotlib`, `PIL` |
| 7 | Error distribution over image statistics | Histograms of brightness/std-dev for correct vs error subsets | `numpy`, `matplotlib` |

## References

| Reference | Link |
|---|---|
| Rules | [LOGGING_CHECKPOINT_RULES.md](../agents/rules/LOGGING_CHECKPOINT_RULES.md), [RESULTS_REPORTING.md](../agents/rules/RESULTS_REPORTING.md) |
| Training script | [src/training/train.py](../src/training/train.py) |
| Model loader | [src/models/dinov3_vit.py](../src/models/dinov3_vit.py) |
| Artifacts | `experiments/results/checkpoints/*/checkpoints/*_best.pt`, `data/splits/test_balanced.csv`, `experiments/results/finetune_report.json` |
| This notebook writes | `experiments/results/error_analysis/` (predictions NPZ, tables CSV, figures PNG) |


In [1]:
# ============================================================
# Step 1: Setup — paths, device, seed, output dirs
# ============================================================
import os, sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT).endswith("notebooks"):
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

# --- Config: Dynamically point at the newly trained model from notebook 02 ---
candidates = [
    # PROJECT_ROOT / "experiments/checkpoints/dinov3_vit_balanced_best.pt",
    # PROJECT_ROOT / "experiments/checkpoints/dinov3_vit_balanced_last.pt",
    PROJECT_ROOT / "experiments/results/checkpoints/20260822_103647_dinov3_finetuned_balanced/checkpoints/dinov3_finetuned_balanced_best.pt"
]
CHECKPOINT = next((c for c in candidates if c.exists()), candidates[0])
TEST_CSV   = PROJECT_ROOT / "data/splits/test_balanced.csv"
MANIFEST   = PROJECT_ROOT / "../data/test_data_v3/manifest.csv"
REPORT     = PROJECT_ROOT / "experiments/results/finetune_report.json"
BACKBONE   = PROJECT_ROOT / "experiments/checkpoints/weights/dinov3-vits16plus-pretrain-lvd1689m/model-3.safetensors"

OUT = PROJECT_ROOT / "experiments/results/error_analysis"
OUT.mkdir(parents=True, exist_ok=True)

if MANIFEST.exists():
    MANIFEST = MANIFEST.resolve()
else:
    MANIFEST = Path("/workspace/data/test_data_v3/manifest.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"Device={device}")
print(f"Selected Model Checkpoint: {CHECKPOINT}")
print(f"Checkpoint exists: {CHECKPOINT.exists()}")
print(f"Test CSV exists: {TEST_CSV.exists()}")
print(f"Manifest exists: {MANIFEST.exists()}")


PROJECT_ROOT=/workspace/hoangtuan/deepfake-ViT
Device=cuda
Selected Model Checkpoint: /workspace/hoangtuan/deepfake-ViT/experiments/results/checkpoints/20260822_103647_dinov3_finetuned_balanced/checkpoints/dinov3_finetuned_balanced_best.pt
Checkpoint exists: True
Test CSV exists: True
Manifest exists: True


In [2]:
# ============================================================
# Step 2: Load model + checkpoint and run predictions ONCE
# Persists predictions to NPZ so later cells are GPU-free.
# ============================================================
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMG_SIZE = 256
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
EVAL_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CsvImageDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.rows = []
        import csv as _csv
        with open(csv_path, newline="") as f:
            for row in _csv.reader(f):
                if len(row) >= 2 and row[0].strip() != "path":
                    self.rows.append((row[0], int(row[1])))
        self.transform = transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        path, label = self.rows[i]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

ds = CsvImageDataset(TEST_CSV, transform=EVAL_TF)
loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
print(f"Test samples: {len(ds)}")

from src.models.dinov3_vit import load_dinov3
from src.training.train import DinoViTClassifier

backbone = load_dinov3(str(BACKBONE), img_size=IMG_SIZE)
model = DinoViTClassifier(backbone).to(device)
ck = torch.load(CHECKPOINT, map_location=device, weights_only=True)
if "model_state_dict" in ck:
    model.load_state_dict(ck["model_state_dict"])
else:
    model.load_state_dict(ck["state_dict"])
model.eval()
print("Checkpoint loaded.")

preds, probs, labels, paths = [], [], [], []
with torch.no_grad():
    for x, y in tqdm(loader, desc="Predicting"):
        x = x.to(device)
        logits = model(x)
        p = torch.softmax(logits, dim=1)
        preds.extend(logits.argmax(1).cpu().tolist())
        probs.extend(p[:, 1].cpu().tolist())
        labels.extend(y.tolist())
        paths.extend([r[0] for r in ds.rows[len(labels)-len(y):len(labels)]])

preds = np.array(preds); probs = np.array(probs); labels = np.array(labels)
paths = np.array(paths)
print(f"Predictions done. preds={preds.shape}, labels={labels.shape}")

np.savez_compressed(OUT / "predictions.npz", preds=preds, probs=probs,
                    labels=labels, paths=paths)
print(f"Saved -> {OUT / 'predictions.npz'}")

Test samples: 4134
Checkpoint loaded.


Predicting: 100%|██████████| 65/65 [00:27<00:00,  2.34it/s]

Predictions done. preds=(4134,), labels=(4134,)
Saved -> /workspace/hoangtuan/deepfake-ViT/experiments/results/error_analysis/predictions.npz


In [3]:
# ============================================================
# Step 2b: Load predictions (no GPU needed for the rest)
# ============================================================
d = np.load(OUT / "predictions.npz")
preds, probs, labels, paths = d["preds"], d["probs"], d["labels"], d["paths"]
print(f"Loaded {len(labels)} samples")
print(f"Positives (fake, label=1): {(labels==1).sum():,} | Negatives (real, label=0): {(labels==0).sum():,}")

Loaded 4134 samples
Positives (fake, label=1): 2,067 | Negatives (real, label=0): 2,067


In [4]:
# ============================================================
# Step 3: Confusion Matrix Deep Dive
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(labels, preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn} FP={fp} FN={fn} TP={tp}  (total={tn+fp+fn+tp})")

if REPORT.exists():
    rep = json.load(open(REPORT))
    rep_cm = np.array(rep["test"]["confusion_matrix"])
    print("\nReported CM (from finetune_report.json):")
    print(rep_cm)
    if rep_cm.sum() == cm.sum():
        print("Reproduced CM matches reported CM.")
    else:
        print(f"WARNING: total differs. Reported={rep_cm.sum()} vs recomputed={cm.sum()}. "
              "The test split was likely regenerated; results below reflect the CURRENT test CSV.")

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real (0)", "Fake (1)"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix — Fine-tuned DINOv3 on test_balanced")
plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=150)
plt.show()

# ---- Error masks ----
fn_mask = (labels == 1) & (preds == 0)   # fakes predicted as real
fp_mask = (labels == 0) & (preds == 1)   # reals predicted as fake
tp_mask = (labels == 1) & (preds == 1)
tn_mask = (labels == 0) & (preds == 0)
print(f"\nError counts -> FN(fake missed)={fn_mask.sum()}, FP(real flagged)={fp_mask.sum()}")

TN=2050 FP=17 FN=234 TP=1833  (total=4134)


KeyError: 'test'

In [6]:
# ============================================================
# Step 4: Error Categorization by metadata (method / domain / identity)
# Joins predictions with test_data_v3/manifest.csv metadata.
# ============================================================
# Build a path -> metadata map from the manifest
meta = {}
if MANIFEST.exists():
    mdf = pd.read_csv(MANIFEST)
    for _, row in mdf.iterrows():
        meta[(row["method"], row["path"])] = {
            "method": row["method"], "domain": row["domain"],
            "identity": row["identity"],
        }

rows = []
for i in range(len(labels)):
    p = paths[i]
    # derive (method, path-rel) from absolute path: .../<method>/fake/<file> or .../real/<file>
    rel = p
    method = None
    low = p.replace("\\", "/")
    parts = low.split("/")
    if "/test_data_v3/" in low:
        idx = parts.index("test_data_v3")
        tail = parts[idx+1:]
        if len(tail) >= 2 and tail[1] == "fake":
            method = tail[0]
            rel = "/".join(tail)
        elif len(tail) >= 1 and tail[0] == "real":
            method = "real"
            rel = "/".join(tail)
    m = meta.get((method, rel), {"method": method, "domain": None, "identity": None})
    rows.append({
        "path": p, "label": int(labels[i]), "pred": int(preds[i]),
        "prob_fake": float(probs[i]),
        "is_error": int(labels[i] != preds[i]),
        "error_type": "FN" if (labels[i]==1 and preds[i]==0) else
                      ("FP" if (labels[i]==0 and preds[i]==1) else "OK"),
        "method": m.get("method"), "domain": m.get("domain"),
        "identity": m.get("identity"),
    })

df = pd.DataFrame(rows)
print(f"Joined {len(df)} samples; {df['method'].notna().sum()} matched to metadata")
df.to_csv(OUT / "error_analysis.csv", index=False)
print(f"Saved -> {OUT / 'error_analysis.csv'}")

Joined 4134 samples; 3122 matched to metadata
Saved -> /workspace/hoangtuan/deepfake-ViT/experiments/results/error_analysis/error_analysis.csv


In [7]:
# ============================================================
# Step 4b: Error rate by method and by domain
# ============================================================
print("=== Error rate by DOMAIN ===")
dom = df.groupby("domain").agg(
    n=("path", "count"), errors=("is_error", "sum"),
    fn=("error_type", lambda s: (s == "FN").sum()),
    fp=("error_type", lambda s: (s == "FP").sum()),
).assign(err_rate=lambda x: x["errors"] / x["n"])
print(dom.round(4).to_string())

print("\n=== Top-12 methods by error rate (only fake methods) ===")
mth = df[df["method"] != "real"].groupby("method").agg(
    n=("path", "count"), fn=("error_type", lambda s: (s == "FN").sum()),
).assign(fn_rate=lambda x: x["fn"] / x["n"])
print(mth.sort_values("fn_rate", ascending=False).head(12).round(4).to_string())

# per-method chart
fig, ax = plt.subplots(figsize=(11, 6))
top = mth.sort_values("fn_rate", ascending=False).head(15)
ax.barh(top.index, top["fn_rate"])
ax.invert_yaxis()
ax.set_xlabel("FN rate (missed fakes / method samples)")
ax.set_title("False-Negative Rate by Deepfake Method")
plt.tight_layout()
plt.savefig(OUT / "fn_by_method.png", dpi=150)
plt.show()

=== Error rate by DOMAIN ===
           n  errors  fn  fp  err_rate
domain                                
cdc      296       1   1   0    0.0034
efs      400      74  74   0    0.1850
fe       203      19  19   0    0.0936
ffc     1355      10   3   7    0.0074
oth      868      17  17   0    0.0196

=== Top-12 methods by error rate (only fake methods) ===
                   n  fn  fn_rate
method                           
MidJourney        44  35   0.7955
whichfaceisreal   51  28   0.5490
styleclip         68  19   0.2794
CollabDiff        43  11   0.2558
MRAA              47   7   0.1489
faceswap          49   5   0.1020
lia               49   4   0.0816
fsgan             41   1   0.0244
hyperreenact      55   1   0.0182
SiT               67   1   0.0149
DiT               80   1   0.0125
sd2.1            100   1   0.0100


In [8]:
# ============================================================
# Step 5: Statistical Breakdown — per-class and per-method metrics
# ============================================================
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Overall
print("=== Overall ===")
print(f"Precision(fake)= {precision_score(labels, preds, zero_division=0):.4f}")
print(f"Recall(fake)=    {recall_score(labels, preds, zero_division=0):.4f}")
print(f"F1(fake)=        {f1_score(labels, preds, zero_division=0):.4f}")
print(f"AUC=             {roc_auc_score(labels, probs):.4f}")

# Per-class
from sklearn.metrics import classification_report
print("\n=== Per-class report ===")
print(classification_report(labels, preds, labels=[0, 1],
                            target_names=["Real", "Fake"], zero_division=0))

# Per-method precision/recall/F1 for fake methods
print("\n=== Per-method detection rate (fake methods) ===")
pm = df[df["method"] != "real"].groupby("method").apply(
    lambda g: pd.Series({
        "n": len(g),
        "detection_rate": (g["pred"] == 1).mean(),
        "FN": (g["error_type"] == "FN").sum(),
    }), include_groups=False
).sort_values("detection_rate")
print(pm.round(4).to_string())

=== Overall ===
Precision(fake)= 0.9908
Recall(fake)=    0.8868
F1(fake)=        0.9359
AUC=             0.9535

=== Per-class report ===
              precision    recall  f1-score   support

        Real       0.90      0.99      0.94      2067
        Fake       0.99      0.89      0.94      2067

    accuracy                           0.94      4134
   macro avg       0.94      0.94      0.94      4134
weighted avg       0.94      0.94      0.94      4134


=== Per-method detection rate (fake methods) ===
                     n  detection_rate    FN
method                                      
MidJourney        44.0          0.2045  35.0
whichfaceisreal   51.0          0.4510  28.0
styleclip         68.0          0.7206  19.0
CollabDiff        43.0          0.7442  11.0
MRAA              47.0          0.8511   7.0
faceswap          49.0          0.8980   5.0
lia               49.0          0.9184   4.0
fsgan             41.0          0.9756   1.0
hyperreenact      55.0          0.9

In [9]:
# ============================================================
# Step 6: Qualitative Analysis — sample misclassified images
# ============================================================
from PIL import Image, ImageDraw, ImageFont

def make_error_grid(mask, title, n=12, ncols=4):
    idx = np.where(mask)[0]
    if len(idx) == 0:
        print(f"No samples for '{title}'")
        return
    # spread selection across confidence: most-confident errors first
    sel = idx[np.argsort(np.abs(probs[idx] - 0.5))][:n] if len(idx) > n else idx
    nrows = int(np.ceil(len(sel) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.2))
    axes = np.atleast_1d(axes).ravel()
    for a in axes:
        a.axis("off")
    for ax, i in zip(axes, sel):
        img = Image.open(paths[i]).convert("RGB")
        ax.imshow(np.array(img))
        m = df.iloc[i]
        tag = f"{m['error_type']} | {m['method']} | p={probs[i]:.2f}"
        ax.set_title(tag, fontsize=8)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    return fig

fig = make_error_grid(fn_mask, "False Negatives — fakes the model called REAL (n=%d)" % fn_mask.sum())
if fig:
    fig.savefig(OUT / "errors_fn.png", dpi=140); plt.show()

fig = make_error_grid(fp_mask, "False Positives — reals the model called FAKE (n=%d)" % fp_mask.sum())
if fig:
    fig.savefig(OUT / "errors_fp.png", dpi=140); plt.show()

NameError: name 'fn_mask' is not defined

In [ ]:
# ============================================================
# Step 6b: Grid-plot ALL samples of the WORST methods (highest FN rate)
# For the methods where detection is weakest, show every test sample so
# the failure mode can be inspected manually.
# ============================================================
from matplotlib import pyplot as plt

# Per-method FN rate (recompute here so it stays in sync with the data)
pm_all = df[df["method"] != "real"].groupby("method").agg(
    n=("path", "count"),
    fn=("error_type", lambda s: (s == "FN").sum()),
    det=("pred", lambda s: (s == 1).mean()),
).assign(fn_rate=lambda x: x["fn"] / x["n"])

# Worst methods = highest FN rate, with at least a few samples to inspect
WORST_N = 4
worst = pm_all.sort_values("fn_rate", ascending=False).head(WORST_N)
print("Worst methods (highest FN rate):")
print(worst.round(3).to_string())

def grid_all_method(method, ncols=4, max_cells=60):
    sub = df[(df["method"] == method) & (df["pred"] == 0)]  # missed fakes
    if sub.empty:
        print(f"[{method}] no FN samples to show")
        return None
    sel = sub.index.tolist()[:max_cells]
    ncols = min(ncols, len(sel))
    nrows = int(np.ceil(len(sel) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.0, nrows * 3.0))
    axes = np.atleast_1d(axes).ravel()
    for a in axes:
        a.axis("off")
    for ax, i in zip(axes, sel):
        img = Image.open(paths[i]).convert("RGB")
        ax.imshow(np.array(img))
        r = df.loc[i]
        ax.set_title(f"p={probs[i]:.2f}\n{r['identity']}", fontsize=8)
    det = float(pm_all.loc[method, "det"])
    fig.suptitle(f"{method} — missed fakes (det_rate={det:.2f}, n_FN={len(sub)})", fontsize=12)
    plt.tight_layout()
    return fig

for method in worst.index:
    fig = grid_all_method(method, ncols=4, max_cells=60)
    if fig is not None:
        safe = method.replace("/", "_")
        fig.savefig(OUT / f"worst_method_{safe}.png", dpi=130)
        plt.show()

Worst methods (highest FN rate):
                  n  fn    det  fn_rate
method                                 
MidJourney       44  35  0.205    0.795
whichfaceisreal  51  28  0.451    0.549
styleclip        68  19  0.721    0.279
CollabDiff       43  11  0.744    0.256


In [ ]:
# ============================================================
# Step 6c: Grid-plot TOP-3 highest-detection methods (all ~100%)
# These are the methods the model detects perfectly; inspecting them
# highlights what "easy" samples look like vs the failing ones above.
# ============================================================
from matplotlib import pyplot as plt

best = pm_all.sort_values("det", ascending=False).head(3)
print("Top-3 detection methods:")
print(best.round(3).to_string())

def grid_method(method, ncols=4, max_cells=48):
    sub = df[df["method"] == method]
    if sub.empty:
        print(f"[{method}] no samples")
        return
    sel = sub.index.tolist()[:max_cells]
    ncols = min(ncols, len(sel))
    nrows = int(np.ceil(len(sel) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.0, nrows * 3.0))
    axes = np.atleast_1d(axes).ravel()
    for a in axes:
        a.axis("off")
    for ax, i in zip(axes, sel):
        img = Image.open(paths[i]).convert("RGB")
        ax.imshow(np.array(img))
        r = df.loc[i]
        ax.set_title(f"p={probs[i]:.2f}\n{r['identity']}", fontsize=8)
    det = float(pm_all.loc[method, "det"])
    fig.suptitle(f"{method} — detected (det_rate={det:.2f})", fontsize=12)
    plt.tight_layout()
    return fig

for method in best.index:
    fig = grid_method(method, ncols=4, max_cells=40)
    if fig:
        safe = method.replace("/", "_")
        fig.savefig(OUT / f"best_method_{safe}.png", dpi=130)
        plt.show()

Top-3 detection methods:
              n  fn  det  fn_rate
method                           
facevid2vid  48   0  1.0      0.0
facedancer   50   0  1.0      0.0
starganv2    69   0  1.0      0.0


In [ ]:
# ============================================================
# Step 7: Error distribution over image statistics
# Brightness / std-dev are cheap proxies for lighting, occlusion,
# compression and background noise. Compare correct vs error subsets.
# ============================================================
def img_stats(paths_subset):
    bright, stdv = [], []
    for p in tqdm(paths_subset, desc="Computing image stats"):
        g = np.array(Image.open(p).convert("L")).astype(np.float32)
        bright.append(g.mean())
        stdv.append(g.std())
    return np.array(bright), np.array(stdv)

ok_mask = labels == preds
err_mask = labels != preds

# subsample for speed if the set is large
def subsample(mask, k=1500):
    idx = np.where(mask)[0]
    rng = np.random.RandomState(0)
    if len(idx) > k:
        idx = rng.choice(idx, k, replace=False)
    return idx

b_ok, s_ok = img_stats(paths[subsample(ok_mask)])
b_err, s_err = img_stats(paths[subsample(err_mask)])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(b_ok, bins=40, alpha=0.6, label="Correct")
axes[0].hist(b_err, bins=40, alpha=0.6, label="Errors")
axes[0].set_xlabel("Mean brightness"); axes[0].set_title("Brightness: Correct vs Errors"); axes[0].legend()
axes[1].hist(s_ok, bins=40, alpha=0.6, label="Correct")
axes[1].hist(s_err, bins=40, alpha=0.6, label="Errors")
axes[1].set_xlabel("Std-dev (contrast/texture)"); axes[1].set_title("Std-dev: Correct vs Errors"); axes[1].legend()
plt.tight_layout()
plt.savefig(OUT / "error_image_statistics.png", dpi=140)
plt.show()

print(f"Brightness: correct {b_ok.mean():.1f}±{b_ok.std():.1f} | errors {b_err.mean():.1f}±{b_err.std():.1f}")
print(f"Std-dev:    correct {s_ok.mean():.1f}±{s_ok.std():.1f} | errors {s_err.mean():.1f}±{s_err.std():.1f}")

Computing image stats: 100%|██████████| 251/251 [00:03<00:00, 72.35it/s]


Brightness: correct 94.4±34.2 | errors 82.9±32.6
Std-dev:    correct 49.9±13.6 | errors 51.3±13.1


In [ ]:
# ============================================================
# Step 8: Export summary + conclusion scaffolding
# ============================================================
summary = {
    "n_test": int(len(labels)),
    "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    "accuracy": float((labels == preds).mean()),
    "precision": float(precision_score(labels, preds, zero_division=0)),
    "recall": float(recall_score(labels, preds, zero_division=0)),
    "f1": float(f1_score(labels, preds, zero_division=0)),
    "roc_auc": float(roc_auc_score(labels, probs)),
}
json.dump(summary, open(OUT / "error_analysis_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))
print("\nArtifacts written to:", OUT)
print("  - predictions.npz, error_analysis.csv, error_analysis_summary.json")
print("  - confusion_matrix.png, fn_by_method.png, errors_fn.png, errors_fp.png, error_image_statistics.png")

print("""
=== Next steps (suggested, based on findings) ===
1. If FN concentrates in a few methods -> check those images (errors_fn.png);
   consider method-level oversampling or a threshold tuned on val.
2. If FP concentrates on 'real' -> tune decision threshold via src/eval/analyze_threshold.py
   (recall/fake-detection trade-off).
3. If errors cluster in low-brightness/high-occlusion -> add brightness/JPEG
   augmentation to the training transform.
4. Report findings with 5W1H per agents/rules/RESULTS_REPORTING.md.
""")

{
  "n_test": 4134,
  "TN": 2050,
  "FP": 17,
  "FN": 234,
  "TP": 1833,
  "accuracy": 0.9392839864537977,
  "precision": 0.9908108108108108,
  "recall": 0.8867924528301887,
  "f1": 0.9359203472044932,
  "roc_auc": 0.9535100031854968
}

Artifacts written to: /workspace/hoangtuan/deepfake-ViT/experiments/results/error_analysis
  - predictions.npz, error_analysis.csv, error_analysis_summary.json
  - confusion_matrix.png, fn_by_method.png, errors_fn.png, errors_fp.png, error_image_statistics.png

=== Next steps (suggested, based on findings) ===
1. If FN concentrates in a few methods -> check those images (errors_fn.png);
   consider method-level oversampling or a threshold tuned on val.
2. If FP concentrates on 'real' -> tune decision threshold via src/eval/analyze_threshold.py
   (recall/fake-detection trade-off).
3. If errors cluster in low-brightness/high-occlusion -> add brightness/JPEG
   augmentation to the training transform.
4. Report findings with 5W1H per agents/rules/RESULTS_R